# RDKit Molecular Descriptor Generation

This notebook generates molecular descriptors for the cleaned JNK3/GSK3β dataset using RDKit. A total of 210 RDKit descriptors are calculated for each molecule and combined with the previously assigned scaffold-based train, validation and test splits. The resulting descriptor dataset is saved as `model_dataset.csv` for subsequent preprocessing and modelling.

In [ ]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

MOLECULE_FILE = "processed_molecules.csv"
SPLIT_FILE = "split_assignments.csv"

molecules_df = pd.read_csv(MOLECULE_FILE)
split_df = pd.read_csv(SPLIT_FILE)

print("Processed molecules:", molecules_df.shape)
print("Split assignments:", split_df.shape)

In [ ]:
data_df = molecules_df.merge(
    split_df,
    on="canonical_smiles",
    how="left"
)

print("Combined dataset shape:", data_df.shape)
print("Missing split assignments:", data_df["split"].isna().sum())

print("\nMolecules in each split:")
print(data_df["split"].value_counts())

In [ ]:
# Prepare the RDKit molecular descriptors

# Obtain the names of all available RDKit 2D descriptors
descriptor_names = [
    descriptor_name
    for descriptor_name, descriptor_function in Descriptors._descList
]

# Creating one calculator that will calculate all descriptors
descriptor_calculator = (
    MoleculeDescriptors.MolecularDescriptorCalculator(
        descriptor_names
    )
)

print("Number of RDKit descriptors:", len(descriptor_names))
print("\nFirst 10 descriptor names:")
print(descriptor_names[:10])

In [ ]:
# Calculate molecular descriptors

descriptor_rows = []

total_molecules = len(data_df)

for position, smiles in enumerate(
    data_df["canonical_smiles"],
    start=1
):
    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        descriptor_values = [np.nan] * len(descriptor_names)
    else:
        try:
            descriptor_values = (
                descriptor_calculator.CalcDescriptors(molecule)
            )
        except Exception:
            descriptor_values = [np.nan] * len(descriptor_names)

    descriptor_rows.append(descriptor_values)

# Show progress after every 10,000 molecules
    if position % 10000 == 0:
        print(
            f"Processed {position} of "
            f"{total_molecules} molecules"
        )

print("Descriptor calculation completed.")

In [ ]:
descriptor_df = pd.DataFrame(
    descriptor_rows,
    columns=descriptor_names
)

# Replace infinite descriptor values with missing values
descriptor_df = descriptor_df.replace(
    [np.inf, -np.inf],
    np.nan
)

model_df = pd.concat(
    [
        data_df.reset_index(drop=True),
        descriptor_df.reset_index(drop=True)
    ],
    axis=1
)

print("Final modelling dataset shape:", model_df.shape)
print("Number of descriptors:", descriptor_df.shape[1])
print(
    "Total missing descriptor values:",
    descriptor_df.isna().sum().sum()
)

OUTPUT_FILE = "model_dataset.csv"

model_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved file:", OUTPUT_FILE)
print("Descriptor dataset saved successfully.")

model_df.head()